### Optional dependency setup
Run this cell only if imports fail in your environment.

In [ ]:
import importlib.util

packages = [
    'numpy', 'matplotlib', 'plotly', 'ipywidgets', 'scipy', 'pandas', 'sklearn', 'seaborn'
]
missing = [p for p in packages if importlib.util.find_spec(p) is None]
if missing:
    print('Installing missing packages:', missing)
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(missing))
else:
    print('All common packages already available.')


In [ ]:
# Dependency bootstrap (auto-install missing packages)
import importlib.util
import subprocess
import sys

PKG_MAP = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'ipywidgets': 'ipywidgets',
}

missing = [pkg for mod, pkg in PKG_MAP.items() if importlib.util.find_spec(mod) is None]
if missing:
    print(f'Installing missing dependencies: {missing}')
    if sys.platform == 'emscripten':
        import micropip
        await micropip.install(missing)
    else:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Dependencies ready.')


# Margin Width Geometry

This notebook visualizes the separating hyperplane together with the two supporting hyperplanes

$$\mathbf{w}^T\mathbf{x} + b = 1, \qquad \mathbf{w}^T\mathbf{x} + b = -1,$$

so you can see directly why the margin width is $2 / \|\mathbf{w}\|$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


In [ ]:
POS = np.array([[-2.8, 2.0], [-2.4, 1.3], [-1.9, 2.7], [-1.2, 0.7], [-0.8, 1.8]])
NEG = np.array([[1.6, -1.4], [2.4, -2.1], [1.9, -0.6], [2.9, -1.2], [0.9, -2.3]])

def line_points(w1, w2, b, c, xs):
    if abs(w2) > 1e-6:
        return xs, -(w1 * xs + b - c) / w2
    return None, None

def plot_margin(w1=1.0, w2=0.8, b=0.0):
    if abs(w1) < 0.15 and abs(w2) < 0.15:
        w1 = 0.15

    xs = np.linspace(-4, 4, 300)
    X1, X2 = np.meshgrid(xs, xs)
    Z = w1 * X1 + w2 * X2 + b
    norm = np.sqrt(w1 ** 2 + w2 ** 2)
    margin_width = 2.0 / norm

    pos_scores = POS @ np.array([w1, w2]) + b
    neg_scores = NEG @ np.array([w1, w2]) + b
    pos_sv = POS[np.argmin(np.abs(pos_scores - 1.0))]
    neg_sv = NEG[np.argmin(np.abs(neg_scores + 1.0))]

    fig, ax = plt.subplots(figsize=(7.2, 7.0))
    ax.contourf(X1, X2, Z, levels=[-20, 0, 20], colors=['#d9e7f5', '#f8e2cc'], alpha=0.95)
    ax.contour(X1, X2, Z, levels=[-1, 1], colors=['#64748b', '#64748b'], linewidths=1.4, linestyles='--')
    ax.contour(X1, X2, Z, levels=[0], colors=['#0f172a'], linewidths=2.0)

    ax.scatter(POS[:, 0], POS[:, 1], c='#2563eb', s=48, label='Class +1', zorder=3)
    ax.scatter(NEG[:, 0], NEG[:, 1], c='#dc2626', s=48, label='Class -1', zorder=3)
    ax.scatter([pos_sv[0], neg_sv[0]], [pos_sv[1], neg_sv[1]], facecolors='none', edgecolors='#111827', s=180, linewidths=2.0, label='Closest to margin', zorder=4)

    mid = 0.5 * (pos_sv + neg_sv)
    ax.annotate('', xy=pos_sv, xytext=neg_sv, arrowprops=dict(arrowstyle='<->', color='#0f766e', lw=2))
    ax.text(mid[0] + 0.15, mid[1] + 0.15, f'Margin width = {margin_width:.2f}', color='#0f766e', fontsize=11, weight='bold')

    ax.text(-3.65, 3.25, r'$\mathbf{w}^T\mathbf{x}+b>0$', fontsize=14, color='#7c2d12')
    ax.text(1.55, -3.35, r'$\mathbf{w}^T\mathbf{x}+b<0$', fontsize=14, color='#1d4ed8')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.grid(alpha=0.18)
    ax.legend(loc='upper right')
    ax.set_title('Interactive SVM Margin Geometry')
    plt.show()
    print(f'||w|| = {norm:.3f} so the margin width is 2/||w|| = {margin_width:.3f}.')

widgets.interact(
    plot_margin,
    w1=widgets.FloatSlider(value=1.0, min=-2.5, max=2.5, step=0.1, description='w1'),
    w2=widgets.FloatSlider(value=0.8, min=-2.5, max=2.5, step=0.1, description='w2'),
    b=widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description='b')
);
